<a href="https://github.com/N3iKos/segsmaker-prallel">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a>

---
**Segsmaker Repo Fusion for Colab**

Run the installer first, then use the form cells below. Empty input slots are skipped automatically.
        


In [ ]:
# @title WebUI Installer {"display-mode":"form"}
# @markdown Pick the WebUI and provide API tokens. `Repository_Raw_Base` must point to the raw URL of this fused repository after it is hosted.
Webui = 'A1111' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Civitai_Key = '' # @param {type:"string", placeholder:"Your Civitai API Key (required)"}
HF_Read_Token = '' # @param {type:"string", placeholder:"Your Hugging Face READ token (optional)"}
Mount_GDrive = 'No' # @param ["Yes", "No"]
Repository_Raw_Base = 'https://raw.githubusercontent.com/N3iKos/segsmaker-prallel/main' # @param {type:"string"}

import subprocess
import sys
from pathlib import Path

if Mount_GDrive == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

_raw_base = Repository_Raw_Base.strip().rstrip('/')
_setup_py = '/content/setup.py'
_setup_url = f'{_raw_base}/script/KC/setup.py'
_result = subprocess.run(['curl', '-fL', '-o', _setup_py, _setup_url], capture_output=True, text=True)
if _result.returncode != 0:
    print(f'Setup script download failed from: {_setup_url}')
    print(_result.stderr)
    sys.exit(1)

get_ipython().run_line_magic(
    'run',
    f'{_setup_py} --webui="{Webui}" --civitai_key="{Civitai_Key}" --hf_read_token="{HF_Read_Token}" --repo_raw_base="{_raw_base}"'
)

if Mount_GDrive == 'Yes':
    drive_root = Path('/content/drive/MyDrive/Segsmaker')
    for name, target in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        if target is None:
            continue
        persistent = drive_root / name
        persistent.mkdir(parents=True, exist_ok=True)
        link = target / f'drive-{name}'
        if not link.exists():
            link.symlink_to(persistent, target_is_directory=True)

    output = drive_root / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    output.mkdir(parents=True, exist_ok=True)
    get_ipython().system(f'rm -rf "{WebUI_Output}"')
    if not WebUI_Output.exists():
        WebUI_Output.symlink_to(output, target_is_directory=True)

    if Webui not in {'ComfyUI', 'SwarmUI'}:
        cache_link = WebUI / 'cache'
        cache_target = drive_root / 'cache'
        cache_target.mkdir(parents=True, exist_ok=True)
        get_ipython().system(f'rm -rf "{cache_link}"')
        if not cache_link.exists():
            cache_link.symlink_to(cache_target, target_is_directory=True)
        


## Model Downloader

Add checkpoint and LoRA links into the slots. Empty slots are skipped. Parallel mode is recommended for multiple files.
        


In [ ]:
# @title Model Downloader - 5 Checkpoints + 5 LoRA + VAE {"display-mode":"form"}
Checkpoint_1 = '' # @param {type:"string", placeholder:"Checkpoint URL or leave empty"}
Checkpoint_2 = '' # @param {type:"string", placeholder:"Checkpoint URL or leave empty"}
Checkpoint_3 = '' # @param {type:"string", placeholder:"Checkpoint URL or leave empty"}
Checkpoint_4 = '' # @param {type:"string", placeholder:"Checkpoint URL or leave empty"}
Checkpoint_5 = '' # @param {type:"string", placeholder:"Checkpoint URL or leave empty"}
Lora_1 = '' # @param {type:"string", placeholder:"LoRA URL or leave empty"}
Lora_2 = '' # @param {type:"string", placeholder:"LoRA URL or leave empty"}
Lora_3 = '' # @param {type:"string", placeholder:"LoRA URL or leave empty"}
Lora_4 = '' # @param {type:"string", placeholder:"LoRA URL or leave empty"}
Lora_5 = '' # @param {type:"string", placeholder:"LoRA URL or leave empty"}
VAE_URL = '' # @param {type:"string", placeholder:"VAE URL or leave empty"}
Parallel_Download = True # @param {type:"boolean"}
Max_Workers = 3 # @param {type:"slider", min:1, max:6, step:1}

from nenen88 import download, parallel_batch_download

_queue = []
for _url in [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]:
    if _url.strip() and CKPT is not None:
        _queue.append((_url.strip(), str(CKPT), None))
for _url in [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]:
    if _url.strip() and LORA is not None:
        _queue.append((_url.strip(), str(LORA), None))
if VAE_URL.strip() and VAE is not None:
    _queue.append((VAE_URL.strip(), str(VAE), None))

if not _queue:
    print('No model URLs provided. Skipping.')
elif Parallel_Download:
    parallel_batch_download(_queue, max_workers=Max_Workers)
else:
    for _url, _dest, _fn in _queue:
        download(f'{_url} {_dest}' + (f' {_fn}' if _fn else ''))
        


## Extra Assets

Optional extensions/custom nodes, embeddings, and upscalers.
        


In [ ]:
# @title Extra Assets {"display-mode":"form"}
Extension_1 = '' # @param {type:"string", placeholder:"Extension/custom node git URL"}
Extension_2 = '' # @param {type:"string", placeholder:"Extension/custom node git URL"}
Extension_3 = '' # @param {type:"string", placeholder:"Extension/custom node git URL"}
Embedding_1 = '' # @param {type:"string", placeholder:"Embedding URL"}
Embedding_2 = '' # @param {type:"string", placeholder:"Embedding URL"}
Upscaler_1 = '' # @param {type:"string", placeholder:"Upscaler URL"}
Upscaler_2 = '' # @param {type:"string", placeholder:"Upscaler URL"}
Parallel_Assets = True # @param {type:"boolean"}
Asset_Workers = 3 # @param {type:"slider", min:1, max:6, step:1}

import tempfile
import os
from nenen88 import clone, download, parallel_batch_download

_extensions = [value.strip() for value in [Extension_1, Extension_2, Extension_3] if value.strip()]
if _extensions and Extensions is not None:
    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as temp:
        temp.write('\n'.join(_extensions))
        temp_path = temp.name
    try:
        get_ipython().run_line_magic('cd', f'-q {Extensions}')
        get_ipython().run_line_magic('clone', temp_path)
    finally:
        os.unlink(temp_path)

_asset_queue = []
for _url in [Embedding_1, Embedding_2]:
    if _url.strip() and Embeddings is not None:
        _asset_queue.append((_url.strip(), str(Embeddings), None))
for _url in [Upscaler_1, Upscaler_2]:
    if _url.strip() and Upscalers is not None:
        _asset_queue.append((_url.strip(), str(Upscalers), None))

if _asset_queue and Parallel_Assets:
    parallel_batch_download(_asset_queue, max_workers=Asset_Workers)
elif _asset_queue:
    for _url, _dest, _fn in _asset_queue:
        download(f'{_url} {_dest}' + (f' {_fn}' if _fn else ''))
else:
    print('No extra assets provided. Skipping.')
        


## FLUX Models

Optional FLUX component downloader. Use only with a compatible WebUI and enough disk space.
        


In [ ]:
# @title FLUX Model Downloader {"display-mode":"form"}
FLUX_Variant = 'None' # @param ["None", "FLUX.1-schnell (Fast, 4-step)", "FLUX.1-dev (Quality, 20-step)"]
FLUX_Unet = 'https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-schnell-fp8.safetensors' # @param {type:"string"}
FLUX_Clip_L = 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors' # @param {type:"string"}
FLUX_T5XXL = 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors' # @param {type:"string"}
FLUX_VAE = 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors' # @param {type:"string"}
FLUX_Workers = 2 # @param {type:"slider", min:1, max:4, step:1}

from nenen88 import parallel_batch_download

if FLUX_Variant == 'None':
    print('FLUX_Variant is None. Skipping.')
else:
    _unet_url = FLUX_Unet.replace('schnell', 'dev') if 'dev' in FLUX_Variant.lower() and 'schnell' in FLUX_Unet else FLUX_Unet
    _te_dest = TE if 'TE' in globals() and TE is not None else CLIP
    _flux_queue = []
    for _url, _dest, _fn in [
        (_unet_url, UNET, None),
        (FLUX_Clip_L, CLIP, None),
        (FLUX_T5XXL, _te_dest, None),
        (FLUX_VAE, VAE, 'flux_ae.safetensors'),
    ]:
        if _url and str(_url).strip() and _dest is not None:
            _flux_queue.append((str(_url).strip(), str(_dest), _fn))

    if _flux_queue:
        parallel_batch_download(_flux_queue, max_workers=FLUX_Workers)
    else:
        print('No compatible FLUX destinations found for this WebUI.')
        


## ControlNet
        


In [ ]:
# @title ControlNet Widget
%run $Controlnet_Widget
        


## Temporary Models

Temporary downloads are stored in `/tmp`-backed model folders and can disappear after the runtime resets.
        


In [ ]:
# @title Temporary Model Downloader {"display-mode":"form"}
TMP_Checkpoint_1 = '' # @param {type:"string", placeholder:"URL [filename] or leave empty"}
TMP_Checkpoint_2 = '' # @param {type:"string", placeholder:"URL [filename] or leave empty"}
TMP_Lora_1 = '' # @param {type:"string", placeholder:"URL [filename] or leave empty"}
TMP_Lora_2 = '' # @param {type:"string", placeholder:"URL [filename] or leave empty"}
TMP_Workers = 3 # @param {type:"slider", min:1, max:6, step:1}

from nenen88 import parallel_batch_download

_tmp_queue = []
for _raw, _dest in [
    (TMP_Checkpoint_1, TMP_CKPT),
    (TMP_Checkpoint_2, TMP_CKPT),
    (TMP_Lora_1, TMP_LORA),
    (TMP_Lora_2, TMP_LORA),
]:
    _parts = _raw.strip().split()
    if not _parts:
        continue
    _tmp_queue.append((_parts[0], str(_dest), _parts[1] if len(_parts) > 1 else None))

if _tmp_queue:
    parallel_batch_download(_tmp_queue, max_workers=TMP_Workers)
else:
    print('No temporary model URLs provided. Skipping.')
        


# Launch
        


In [ ]:
# @title Launch WebUI {"display-mode":"form"}
Launch_Profile = 'Default' # @param ["Default", "Low VRAM", "Medium VRAM", "No half", "Custom only"]
Extra_Args = '' # @param {type:"string", placeholder:"Additional launch arguments"}
Skip_Widget = True # @param {type:"boolean"}
Skip_ComfyUI_Check = False # @param {type:"boolean"}
NGROK_Token = '' # @param {type:"string", placeholder:"Optional NGROK token"}
ZROK_Token = '' # @param {type:"string", placeholder:"Optional ZROK token"}

_profile_args = {
    'Default': '',
    'Low VRAM': '--lowvram',
    'Medium VRAM': '--medvram',
    'No half': '--no-half --precision full',
    'Custom only': '',
}
_args = ' '.join(part for part in [_profile_args.get(Launch_Profile, ''), Extra_Args.strip()] if part).strip()
if Skip_ComfyUI_Check:
    _args = f'--skip-comfyui-check {_args}'.strip()
if NGROK_Token.strip():
    _args = f'--N "{NGROK_Token.strip()}" {_args}'.strip()
if ZROK_Token.strip():
    _args = f'--Z "{ZROK_Token.strip()}" {_args}'.strip()

get_ipython().run_line_magic('cd', f'-q {WebUI}')
get_ipython().run_line_magic('run', f'segsmaker.py {_args}')
        
